# 05 - Evaluate Reactant and Condition Models

Loads both LoRA adapters, runs held-out tests, and writes JSON reports.

In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes datasets huggingface_hub safetensors sentencepiece rdkit tqdm

import json
import re
import time
from pathlib import Path
from typing import Any, Optional

import torch
from datasets import load_dataset
from peft import PeftModel
from rdkit import Chem, RDLogger
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

RDLogger.DisableLog('rdApp.warning')
RDLogger.DisableLog('rdApp.error')

BASE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
REACTANT_LORA_ID = 'oleh13/retro-reactants-qwen25-7b-lora'
CONDITION_LORA_ID = 'oleh13/retro-conditions-qwen25-7b-lora'
WORK_DIR = Path('/content/retro_condition_agent')
DATA_DIR = WORK_DIR / 'data'
REPORT_DIR = WORK_DIR / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

REACTANTS_TEST_JSONL = DATA_DIR / 'reactants_test.jsonl'
CONDITIONS_TEST_JSONL = DATA_DIR / 'conditions_test.jsonl'
MAX_TEST_ROWS = 300
DEVICE_MAP = 'auto'

assert REACTANTS_TEST_JSONL.exists(), f'Missing {REACTANTS_TEST_JSONL}'
assert CONDITIONS_TEST_JSONL.exists(), f'Missing {CONDITIONS_TEST_JSONL}'


In [ ]:
def canonical_smiles(smiles: Optional[str]) -> Optional[str]:
    if not smiles:
        return None
    try:
        mol = Chem.MolFromSmiles(str(smiles).strip())
        return Chem.MolToSmiles(mol, canonical=True) if mol is not None else None
    except Exception:
        return None

def canon_set(values) -> set[str]:
    if not isinstance(values, list):
        return set()
    return {can for can in (canonical_smiles(v) for v in values) if can}

def extract_json(text: str) -> dict:
    text = text.strip()
    text = re.sub(r'^```(?:json)?', '', text, flags=re.I).strip()
    text = re.sub(r'```$', '', text).strip()
    first, last = text.find('{'), text.rfind('}')
    if first == -1 or last == -1 or last <= first:
        raise ValueError('no JSON object found')
    return json.loads(text[first:last + 1])

def load_jsonl(path: Path, limit: int | None = None) -> list[dict]:
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            rows.append(json.loads(line))
            if limit and len(rows) >= limit:
                break
    return rows

def expected_assistant(record: dict) -> dict:
    return json.loads(record['messages'][2]['content'])

def prompt_messages(record: dict) -> list[dict[str, str]]:
    return record['messages'][:2]

def load_lora_model(adapter_id: str):
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map=DEVICE_MAP, torch_dtype=torch.bfloat16, trust_remote_code=True)
    model = PeftModel.from_pretrained(base, adapter_id)
    model.eval()
    return tokenizer, model

def generate_json(tokenizer, model, messages: list[dict[str, str]], max_new_tokens: int = 256) -> tuple[dict | None, str, str | None]:
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([prompt], return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None, eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id)
    raw = tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()
    try:
        return extract_json(raw), raw, None
    except Exception as exc:
        return None, raw, str(exc)


In [ ]:
reactant_records = load_jsonl(REACTANTS_TEST_JSONL, MAX_TEST_ROWS)
condition_records = load_jsonl(CONDITIONS_TEST_JSONL, MAX_TEST_ROWS)
samples_path = REPORT_DIR / 'sample_predictions.jsonl'

react_tok, react_model = load_lora_model(REACTANT_LORA_ID)
react_stats = {'total': 0, 'json_valid': 0, 'valid_smiles': 0, 'exact_reactant_match': 0, 'reaction_class_match': 0, 'product_as_sole_reactant': 0}
samples = []

for record in tqdm(reactant_records, desc='reactant eval'):
    expected = expected_assistant(record)
    user = json.loads(record['messages'][1]['content'])
    pred, raw, err = generate_json(react_tok, react_model, prompt_messages(record), max_new_tokens=256)
    react_stats['total'] += 1
    if pred is not None:
        react_stats['json_valid'] += 1
        predicted_set = canon_set(pred.get('reactants'))
        expected_set = canon_set(expected.get('reactants'))
        if predicted_set and len(predicted_set) == len(pred.get('reactants', [])):
            react_stats['valid_smiles'] += 1
        if predicted_set == expected_set:
            react_stats['exact_reactant_match'] += 1
        if str(pred.get('reaction_class')) == str(expected.get('reaction_class')):
            react_stats['reaction_class_match'] += 1
        product = canonical_smiles(user.get('product_smiles'))
        if product and predicted_set == {product}:
            react_stats['product_as_sole_reactant'] += 1
    samples.append({'task': 'reactants', 'input': user, 'expected': expected, 'prediction': pred, 'raw': raw, 'error': err})

react_report = {k: v for k, v in react_stats.items()}
for key in ['json_valid', 'valid_smiles', 'exact_reactant_match', 'reaction_class_match', 'product_as_sole_reactant']:
    react_report[key + '_rate'] = react_stats[key] / max(1, react_stats['total'])
(REPORT_DIR / 'reactant_eval.json').write_text(json.dumps(react_report, indent=2), encoding='utf-8')
print(json.dumps(react_report, indent=2))

del react_model
torch.cuda.empty_cache()


In [ ]:
cond_tok, cond_model = load_lora_model(CONDITION_LORA_ID)
condition_stats = {'total': 0, 'json_valid': 0, 'has_solvent_or_reagents': 0, 'temperature_has_units': 0, 'time_has_units': 0, 'fake_evidence_ids': 0}
unit_re = re.compile(r'(c|°c|deg|room temperature|rt|reflux|ambient|h|hr|hour|min|minute|overnight)', re.I)

for record in tqdm(condition_records, desc='condition eval'):
    expected = expected_assistant(record)
    user = json.loads(record['messages'][1]['content'])
    pred, raw, err = generate_json(cond_tok, cond_model, prompt_messages(record), max_new_tokens=320)
    condition_stats['total'] += 1
    if pred is not None:
        condition_stats['json_valid'] += 1
        if pred.get('solvent') not in (None, '', 'not specified') or pred.get('reagents') not in (None, '', 'not specified'):
            condition_stats['has_solvent_or_reagents'] += 1
        if unit_re.search(str(pred.get('temperature_celsius', ''))):
            condition_stats['temperature_has_units'] += 1
        if unit_re.search(str(pred.get('reaction_time', ''))):
            condition_stats['time_has_units'] += 1
        if pred.get('evidence_reaction_ids'):
            condition_stats['fake_evidence_ids'] += 1
    samples.append({'task': 'conditions', 'input': user, 'expected': expected, 'prediction': pred, 'raw': raw, 'error': err})

condition_report = {k: v for k, v in condition_stats.items()}
for key in ['json_valid', 'has_solvent_or_reagents', 'temperature_has_units', 'time_has_units', 'fake_evidence_ids']:
    condition_report[key + '_rate'] = condition_stats[key] / max(1, condition_stats['total'])
(REPORT_DIR / 'condition_eval.json').write_text(json.dumps(condition_report, indent=2), encoding='utf-8')
with samples_path.open('w', encoding='utf-8') as f:
    for item in samples:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')
print(json.dumps(condition_report, indent=2))
print('saved reports:', REPORT_DIR)
